In [9]:

import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()
myJekyllDir = '/Users/AJ/Github_AJ/garcia9810.github.io/assets/json/'

In [10]:
licenses_url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/licenses_fall2022.csv" 
licenses = pd.read_csv(licenses_url)
licenses['Effective Date'] = pd.to_datetime(
    licenses['Effective Date'],
    errors='coerce'
)

licenses = licenses.dropna(subset=['Effective Date'])
licenses['year'] = licenses['Effective Date'].dt.year
licenses['month'] = licenses['Effective Date'].dt.month
licenses['APPLICATION_RECEIVED_DATE'] = licenses['Effective Date']
licenses['LICENSE_DESCRIPTION'] = licenses['Description'] 

license_counts = (
    licenses['LICENSE_DESCRIPTION']
    .value_counts()
    .reset_index()
)

license_counts.columns = ['LICENSE_DESCRIPTION', 'count']
license_counts = license_counts.head(15)

In [11]:
chart1 = (
    alt.Chart(license_counts)
    .mark_bar()
    .encode(
        alt.X('count:Q', title='Number of Licenses'),
        alt.Y('LICENSE_DESCRIPTION:N',
              sort='-x',
              title='License Type'),
        alt.Color('count:Q',
                  legend=None,
                  title='Count'),
        tooltip=[
            alt.Tooltip('LICENSE_DESCRIPTION:N', title='License Type'),
            alt.Tooltip('count:Q', title='Count')
        ]
    )
    .properties(
        width=450,
        height=320,
        title='Top 15 Most Common License Types in Fall 2022'
    )
)




In [12]:
chart1.save(myJekyllDir + 'licenses_top_types.json')


In [13]:
brush = alt.selection_interval(encodings=['x'])
chart_months = (
    alt.Chart(licenses)
    .mark_bar()
    .encode(
        alt.X('month:Q', bin=False, title='Application Month'),
        alt.Y('count()', title='Number of Applications')
    )
    .properties(
        width=500,
        height=150,
        title='Licenses by Month — Brush to Filter'
    )
    .add_params(brush)
)

chart_types_filtered = (
    alt.Chart(licenses)
    .mark_bar()
    .encode(
        alt.X('count():Q', title='Number of Applications'),
        alt.Y('LICENSE_DESCRIPTION:N',
              sort='-x',
              title='License Type'),
        alt.Color('LICENSE_DESCRIPTION:N', legend=None),
        tooltip=[
            alt.Tooltip('LICENSE_DESCRIPTION:N', title='License Type'),
            alt.Tooltip('count():Q', title='Count')
        ]
    )
    .transform_filter(brush)
    .properties(
        width=500,
        height=300,
        title='License Types Within Selected Month Range'
    )
)


In [14]:
chart2 = chart_months & chart_types_filtered



In [15]:
chart2.save(myJekyllDir + 'licenses_month_linked.json')
